## 2) Convert Coco to ODVG

In [1]:
import re

In [1]:
import re

# Define the file path
file_path = '/GroundingDINO-Med-main/OpenGroundingDino/tools/coco2odvg.py'

# Define the new values according to the dataset
new_id_map = '{0: 0, 1:1, 2:2}'
#new_ori_map = '{"0": "vertebra"}'
#new_ori_map = '{"0": "entrance"}'
#new_ori_map = '{"0": "fire", "1":"smoke"}'
new_ori_map = '{"0": "door symbol", "1": "window symbol", "2": "zone"}'
#new_ori_map = '{"0": "jellyfish", "1": "penguin", "2": "shark", "3": "starfish", "4": "stingray"}'

# Read the content of the file
with open(file_path, 'r') as file:
    content = file.read()

# Replace the id_map value using regex
content = re.sub(r'id_map\s*=\s*\{[^\}]*\}', f'id_map = {new_id_map}', content)

# Replace the ori_map value using regex
content = re.sub(r'ori_map\s*=\s*\{[^\}]*\}', f'ori_map = {new_ori_map}', content)

# Write the updated content back to the file
with open(file_path, 'w') as file:
    file.write(content)

print(f"Updated {file_path} successfully.")

Updated /GroundingDINO-Med-main/OpenGroundingDino/tools/coco2odvg.py successfully.


## 3) Convert COCO JSON to trainable parameter for grounding dino

In [ ]:
#!pip install jsonlines

In [ ]:
# change path of input file to your input Coco json file

!python /GroundingDINO-Med-main/OpenGroundingDino/tools/coco2odvg.py --input "/GroundingDINO-Med-main/floor_plans/sample5/merged/train/_annotations.coco.json"  --output "/GroundingDINO-Med-main/input_params/train.jsonl"

loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
  == dump meta ...
  == done.



100%|██████████| 5/5 [00:00<?, ?it/s]


In [3]:
import json

# Define the content for the JSON file
#content = {"0": "jellyfish", "1": "penguin", "2": "shark", "3": "starfish", "4": "stingray"}
content = {"0": "door symbol", "1": "window symbol", "2": "zone"}
#content = {"0": "entrance"}
#content = {"0": "fire", "1":"smoke"}
#content = {"0": "vertebra"}
# Define the file path
file_path = '/GroundingDINO-Med-main/input_params/label.json'

# Write the content to the JSON file
with open(file_path, 'w') as file:
    json.dump(content, file)

print(f"File '{file_path}' created successfully.")

File '/GroundingDINO-Med-main/input_params/label.json' created successfully.


In [4]:
#change the paths according to file locations
import json

# Define the data
data = {
    "train": [
        {
            "root": "/GroundingDINO-Med-main/floor_plans/sample5/merged/train",#Train images
            "anno": "/GroundingDINO-Med-main/input_params/train.jsonl",#Odvg jsonl file
            "label_map": "/GroundingDINO-Med-main/input_params/label.json",# label.json file
            "dataset_mode": "odvg"
        }
    ],
    "val": [
        {
            "root": "/GroundingDINO-Med-main/floor_plans/sample/merged/valid",#Test Images
            "anno": "/GroundingDINO-Med-main/floor_plans/sample/merged/valid/_annotations.coco.json",#Test data Annotation file in COCO
            "label_map": None,
            "dataset_mode": "coco"
        }
    ]
}

file_path = '/GroundingDINO-Med-main/OpenGroundingDino/config/datasets_mixed_odvg.json'

with open(file_path, 'w') as file:
    json.dump(data, file, indent=2)

print(f"Data has been written to {file_path}")

Data has been written to /GroundingDINO-Med-main/OpenGroundingDino/config/datasets_mixed_odvg.json


In [5]:
import re

def modify_file(file_path):
    #label_list_content = 'label_list = ["jellyfish", "penguin", "shark", "starfish", "stingray"]\n'
    label_list_content = 'label_list = ["door symbol", "window symbol", "zone"]\n'
    #label_list_content = 'label_list = ["entrance"]\n'
    #label_list_content = 'label_list = ["fire", "smoke"]\n'
    #label_list_content = 'label_list = ["vertebra"]\n'
    # Read the entire content of the file
    with open(file_path, 'r') as file:
        content = file.read()

    # Replace use_coco_eval =TRUE with use_coco_eval =FALSE using regex
    content = re.sub(r'use_coco_eval\s*=\s*True', 'use_coco_eval = False', content)

    # Insert label_list after use_coco_eval = FALSE using regex
    content = re.sub(r'use_coco_eval\s*=\s*False', r'use_coco_eval = False\n\n' + label_list_content, content, count=1, flags=re.MULTILINE)

    # Write the modified content back to the file
    with open(file_path, 'w') as file:
        file.write(content)

# Paths to the files
cfg_coco_path = '/GroundingDINO-Med-main/OpenGroundingDino/config/cfg_coco.py'
cfg_odvg_path = '/GroundingDINO-Med-main/OpenGroundingDino/config/cfg_odvg.py'

# Modify both files
modify_file(cfg_coco_path)
modify_file(cfg_odvg_path)

print("Updated use_coco_eval to FALSE and added label_list using regex in both files.")

Updated use_coco_eval to FALSE and added label_list using regex in both files.


In [20]:
from transformers import AutoTokenizer, AutoModel

save_dir = "C:/GroundingDINO-Med-main/bert/"

tokenizer = AutoTokenizer.from_pretrained("dmis-lab/biobert-base-cased-v1.1")
model = AutoModel.from_pretrained("dmis-lab/biobert-base-cased-v1.1")

tokenizer.save_pretrained(save_dir)
model.save_pretrained(save_dir)

print(f"BioBERT tokenizer and model saved in {save_dir}")

BioBERT tokenizer and model saved in C:/GroundingDINO-Med-main/bert/


In [13]:
print(model.config._name_or_path)

dmis-lab/biobert-base-cased-v1.1


## 3) Train the model using around 20 images per class

In [27]:
import torch

torch.cuda.empty_cache()

In [ ]:
python "C:/GroundingDINO-Med-main/OpenGroundingDino/main.py" --config_file "C:/GroundingDINO-Med-main/OpenGroundingDino/config/cfg_odvg.py" --datasets "C:/GroundingDINO-Med-main/OpenGroundingDino/config/datasets_mixed_odvg.json" --output_dir "C:/GroundingDINO-Med-main/output/sample5" --pretrain_model_path "C:/GroundingDINO-Med-main/groundingdino_swint_ogc.pth"

SyntaxError: invalid syntax (3033329425.py, line 1)

## 4) Evaulates the result for test images using the trained model

In [8]:
import os
import subprocess

# Directory containing the images
image_dir = "/GroundingDINO-Med-main/xray_data/sample_20/merged/valid"

# Get a list of all image files in the directory
image_files = [f for f in os.listdir(image_dir) if f.endswith('.png') or f.endswith('.jpg')]

# Define the other arguments for the inference script
config_path = "C:/GroundingDINO-Med-main/OpenGroundingDino/tools/GroundingDINO_SwinT_OGC.py"
checkpoint_path = "C:/GroundingDINO-Med-main/output/checkpoint0019.pth"  # Change
text_prompts = "aortic enlargement . pneumothorax"
output_dir = "C:/GroundingDINO-Med-main/pred_images/"

# Loop over all image files and run the inference script on each one
for image_file in image_files:
    image_path = os.path.join(image_dir, image_file)
    output_path = os.path.join(output_dir, image_file)
    command = [
        "python", "C:/GroundingDINO-Med-main/OpenGroundingDino/tools/inference_on_a_image.py",
        "-c", config_path,
        "-p", checkpoint_path,
        "-i", image_path,
        "-t", text_prompts,
        "-o", output_path
    ]
    subprocess.run(command)

KeyboardInterrupt: 

In [20]:
!python "C:/GroundingDINO-Med-main/OpenGroundingDino/inference_on_a_image.py" -c "C:\GroundingDINO-Med-main\OpenGroundingDino\tools\GroundingDINO_SwinT_OGC.py" -p "C:/GroundingDINO-Med-main/output1/checkpoint0019.pth" -i "C:/GroundingDINO-Med-main/xray_data/sample_20/merged/test/af201da8a5f8354c4c3291995d5cbafd.jpg" -t "aortic enlargement . pneumothorax " -o "C:/GroundingDINO-Med-main/pred_images/"

final text_encoder_type: bert-base-uncased
load tokenizer done.
<All keys matched successfully>
Predictions saved to C:/GroundingDINO-Med-main/pred_images/af201da8a5f8354c4c3291995d5cbafd_predictions.json


c:\Users\limfo\.conda\envs\opendino1\lib\site-packages\timm\models\layers\__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
c:\Users\limfo\.conda\envs\opendino1\lib\site-packages\torch\functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\TensorShape.cpp:4319.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
c:\Users\limfo\.conda\envs\opendino1\lib\site-packages\transformers\modeling_utils.py:1621: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(
c:\Users\limfo\.conda\envs\opendino1\lib\site-packages\torch\_dynamo\eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_r

## 5) Finetune the result by adjusting the parameters 

## 6) Evaulates the result using zero shot using the finetuned model.